# 02 - Demand Forecasting
Direct multi-horizon forecasting: one model per horizon (7/15/30 days) predicts the TOTAL units sold over the next h days from features known at the forecast origin. Chronological split, validation-based model selection, baselines first.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import config
from IPython.display import Image, display, Markdown
from src import demand_forecasting as dfc
from src.feature_engineering import build_demand_features, FEATURE_COLUMNS
import matplotlib.pyplot as plt

In [ ]:
sales = pd.read_csv(config.PROCESSED_DIR/'sales_daily_clean.csv', parse_dates=['Date'])
meds = pd.read_csv(config.PROCESSED_DIR/'medicines_clean.csv')
feat, enc = build_demand_features(sales, meds)
feat[['Date','Medicine_ID','Units_Sold']+FEATURE_COLUMNS[6:11]+['y_7','y_15','y_30']].dropna().head()

## Leakage check
Split boundaries for the 7-day horizon: training targets must end before the test period starts.

In [ ]:
sp = dfc.split_by_origin(feat, 7)
print('train origins up to', sp.train.Date.max().date(), '-> targets end', (sp.train.Date.max()+pd.Timedelta(days=7)).date())
print('test origins start ', sp.test.Date.min().date())

## Train and compare (takes ~1-2 minutes)

In [ ]:
res = dfc.train_demand_models(sales, meds, n_estimators=100, save=False, verbose=False)
m = res['metrics']
m[m.Split=='test'].sort_values(['Horizon','MAE']).round(2)

In [ ]:
# Actual vs predicted for one medicine (7-day horizon, test period)
p = res['test_predictions']; mid = p.Medicine_ID.iloc[0]
q = p[(p.Medicine_ID==mid)&(p.Horizon==7)]
plt.figure(figsize=(10,4)); plt.plot(q.Origin_Date,q.Actual,label='Actual'); plt.plot(q.Origin_Date,q.Predicted,'--',label='Predicted'); plt.legend(); plt.title(f'{mid}: units over next 7 days'); plt.show()